Things to add/areas for improvement
- simulation time step -> 1hr?, 30 mins?
      - smart grid smart city data uses 30 min intervals for household demand/usage, 48 timesteps per simulation scenario to simulate a full day
      - ensure generation profiles are using half hourly TMY data in SAM to match
-  realistic load/generation profiles for houses
      - solar curves
      - battery charge/discharge profliles
      - EV charge/discharge profiles -> dynamic and handled by the simulation rather than predefined profiles
      - these might be found from realworld sources/data tables, find smart meter data if possible -> found smart grid smart city meter data need to clean and select house profiles
      - should consider days where temperature and weather changes increase demand or decrease generation
            - check smart grid smart city data for location and date information could cross check with TMY data for that year/location
            - generation profiles can be handled in SAM
- realistic load profiles for industry/hospital
      - try and find realworld data to ensure daily curves are accurate
      - need more research to ensure realism
- Improve Windturbine simulation
      - Different weather profiles that show no wind/lockout, moderate and ideal wind (can use to show DFIG behaviour)
- can this each simulation run based on changning input settings?
      -confident the code so far can be adapted later to take inputs from external sources i.e buttons or removal of EV models
- data outputs that can be represented graphically and physically with leds etc
     - night/day time shift and change -> can this be visibly shown on display? e.g street lamps etc
     - 
- increasing complexity of power flow -> add pandapower validation layer
     - add things like power factor, transmission lines, transformers/substations and how these effect the simulation
     - how do inverters and their powerfactors effect the simulation
     - this will likely be handled by pandapower
- ADD VPP controller
     - start with controlling basic simulation layer
     - then add inputs and response from pandapower validation layer
     - Stage 1:
        Predefined event hours

     - Stage 2:
        Dataset-driven events
        hospital_load_kw exceeds threshold

     - Stage 3:
        User-input-driven events
        button/switch changes external_event

     - Stage 4:
        Network-driven events
        pandapower detects voltage/loading issue

     - Stage 5:
        Market-driven events
        price signal changes mode
       
- Financial moddelling
     - import/export tarif calculations
     - research indicates that VPPs allow for hoigher export tarifs than traditional residential PV exports -> integrate both models and show difference on data screen?


- Current workflow goals
    - Put the existing loop into VPPSimulation.step().
    - Use modulo indexing so the daily profile repeats.
    - Add current_step, elapsed time, day, and hour fields.
    - add more explicit heirachy to the house controllers (inverters)
    - add varying weather/generation/load profiles 
    - Let step() receive an external_inputs dictionary.
    - Update VPPController.get_signals() to support manual overrides.
    - Add start, pause, reset, and export methods.
    - Add a rolling results buffer for live graphing.
    - Add condition-based controller triggers.
    - Add targeted battery dispatch for local-load support.

1. **Create a reusable simulation engine**
   Move the existing loop into a VPPSimulation class with a step() method that advances the simulation by one timestep and returns the latest system state.

2. **Enable continuous Simulation**
   Use modulo indexing so 24-hour or 48-step daily load and generation profiles repeat while the overall simulation continues indefinitely.

3. **Add simulation time tracking**
   Record the total simulation step, elapsed simulated time, day number, profile index, and hour-of-day.

4. **Establish the control hierarchy**
   Use a three-level structure:

        - Central VPP controller determines the overall operating objective.
        - Household/local controller determines how much each house can contribute.
        - Battery/inverter model enforces physical limits such as SOC, charge/discharge power, and efficiency.

5. **Accept dynamic external inputs**
    Pass an external_inputs dictionary into each simulation step so buttons, switches, network-support requests, and other demonstrator inputs can affect operation while the simulation is running.
   
6. **Define controller priority rules**
   Establish a clear order for emergency overrides, manual commands, local events, peak shaving, cheap-energy charging, and default operation.

7. **Separate commands from operating conditions**
   Allow the controller to respond both to direct user commands and to simulation conditions such as high grid import, high local demand, or low battery SOC.

8. **improve VPP dispatch Logic**
    For local-load or network-support events, calculate the required VPP power contribution, determine available resources, and allocate dispatch targets across participating households.

9. **Add Simulation Controls**
    Provide start, pause, reset, and single-step functionality. Resetting should also restore battery SOC and any other stateful components to their initial conditions.

10. **Add live-results handling**
   Maintain a rolling results buffer for real-time graphs while optionally storing the full simulation history for analysis and export.

11. **eparate export from simulation execution**
   Keep results in memory during operation and export to Excel or another format only when requested.

12. **Keep timing outside the simulation engine**
    Let the eventual user interface or demonstrator timer decide when step() is called, preventing the simulation from blocking the interface.




In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import os


#create results folder if it doesn't exist
os.makedirs("results", exist_ok=True)


print("Environment working")


Environment working


Define simulation Objects


In [25]:
class VPPSimulation:
    def __init__(self,
                 houses,
                 hospital,
                 wind,
                 controller,
                 timestep_hours,
                 num_steps
                ):
        
        self.houses = houses
        self.hospital = hospital
        self.wind = wind
        self.controller = controller

        self.timestep_hours = timestep_hours
        self.num_steps = num_steps

        self.current_step = 0

        self.simulation_results_summary = []

        self.house_results_log = {
            house.name: [] for house in houses
        }
    def step(self, external_inputs=None):
        if external_inputs is None:
            
            external_inputs = {}
            
        profile_index = self.current_step % self.num_steps

        house_states = [
            house.get_state(
                profile_index,
                self.timestep_hours
                )
                for house in self.houses
        ]
        
        signals = self.controller.get_signals(
            profile_index,
            external_inputs,
            house_states
        )
  
            

        house_results = [
            house.step(
                profile_index,
                self.timestep_hours,
                signals
            )
            for house in self.houses
        ]

        hospital_result = self.hospital.step(profile_index)
        wind_result = self.wind.step(profile_index)
                # Aggregate results from all houses
        total_house_demand_kw = sum(
            h["load_kw"] for h in house_results
        )

        total_house_generation_kw = sum(
            h["generation_kw"] for h in house_results
        )

        total_house_net_power_kw = sum(
            h["net_power_kw"] for h in house_results
        )

        total_house_net_power_before_battery_kw = sum(
            h["net_power_kw_before_battery"] for h in house_results
        )

        total_battery_power_kw = sum(
            h["battery_power_kw"] for h in house_results
        )

        total_battery_soc_kwh = sum(
            h["battery_soc_kwh"] for h in house_results
        )

        total_battery_discharge_kw = sum(
            h["battery_discharge_kw"] for h in house_results
        )

        total_battery_charge_kw = sum(
            h["battery_charge_kw"] for h in house_results
        )

        total_local_load_kw = hospital_result["load_kw"]
        total_wind_generation_kw = wind_result["generation_kw"]


        # Calculate total grid power
        net_grid_power_kw = (
            total_house_net_power_kw
            + total_local_load_kw
            - total_wind_generation_kw
        )

        grid_import_kw = max(net_grid_power_kw, 0.0)
        grid_export_kw = -min(net_grid_power_kw, 0.0)


        # Calculate simulation time
        elapsed_hours = self.current_step * self.timestep_hours
        day_number = int(elapsed_hours // 24) + 1
        hour_of_day = elapsed_hours % 24


        # Build result for this simulation step
        timestep_result = {
            "Simulation_step": self.current_step,
            "Profile_index": profile_index,
            "Elapsed_hours": elapsed_hours,
            "Day_number": day_number,
            "Hour_of_day": hour_of_day,

            "House_demand_kw": total_house_demand_kw,
            "House_generation_kw": total_house_generation_kw,
            "House_net_power_before_battery_kw":
                total_house_net_power_before_battery_kw,

            "Battery_power_kw": total_battery_power_kw,
            "Battery_soc_kwh": total_battery_soc_kwh,
            "Battery_discharge_kw": total_battery_discharge_kw,
            "Battery_charge_kw": total_battery_charge_kw,
            "Available_VPP_discharge_kw": signals["total_available_discharge_kw"],

            "Battery_discharge_kwh":
                total_battery_discharge_kw * self.timestep_hours,

            "Battery_charge_kwh":
                total_battery_charge_kw * self.timestep_hours,

            "House_net_power_kw": total_house_net_power_kw,
            "Hospital_load_kw": total_local_load_kw,
            "Wind_generation_kw": total_wind_generation_kw,

            "Net_grid_power_kw": net_grid_power_kw,
            "Grid_import_kw": grid_import_kw,
            "Grid_export_kw": grid_export_kw,

            "VPP_mode": signals["mode"],
            "Is_peak_period": signals["is_peak_period"],
            "Is_event_period": signals["is_event_period"],
            "Is_cheap_energy_period":signals["is_cheap_energy_period"],

            "Requested_support_kw": signals["requested_support_kw"],
            "Available_VPP_discharge_kw":signals["total_available_discharge_kw"],
        }


        # Store summary result
        self.simulation_results_summary.append(timestep_result)


        # Store each individual house result
        for result in house_results:
            house_name = result["name"]
            self.house_results_log[house_name].append(result)


        # Advance simulation clock
        self.current_step += 1


        # Return the latest result
        return timestep_result


class House:
    def __init__(self, 
                 name, 
                 load_profile, 
                 generation_profile, 
                 battery=None
                ):
        
        self.name = name
        self.load_profile = load_profile
        self.generation_profile = generation_profile
        self.battery = battery

        
    def get_state(self, profile_index, timestep_hours):
        load_kw = self.load_profile[profile_index]
        generation_kw = self.generation_profile[profile_index]

        net_power_kw_before_battery = load_kw - generation_kw
        battery_soc_kwh = None
        battery_soc_percent = None
        max_discharge_kw = 0.0
        max_charge_kw = 0.0
        available_discharge_kw = 0.0

        if self.battery is not None:

            battery_soc_kwh = self.battery.get_soc_kwh()
            battery_soc_percent = self.battery.get_soc_percent()

            max_discharge_kw = self.battery.max_discharge_kw
            max_charge_kw = self.battery.max_charge_kw

            available_energy_kwh = max(
                
                0.0,
                self.battery.soc_kwh - self.battery.min_soc_kwh
            )
            available_discharge_kw = min(
                self.battery.max_discharge_kw,
                available_energy_kwh * self.battery.efficiency / timestep_hours
            )

        return {
                "name": self.name,
                "load_kw": load_kw,
                "generation_kw": generation_kw,
                "net_power_kw_before_battery": net_power_kw_before_battery,

                "has_battery": self.battery is not None,
                
                "battery_soc_kwh": battery_soc_kwh,
                "battery_soc_percent": battery_soc_percent,
                
                "max_discharge_kw": max_discharge_kw,
                "max_charge_kw": max_charge_kw,

                "available_discharge_kw": available_discharge_kw,
            }
            
                
                

    def step(self, t, timestep_hours, signals):
        load_kw = self.load_profile[t]
        generation_kw = self.generation_profile[t]
        # check if there is a surplus of generation
        net_power_kw_bb = load_kw - generation_kw
        # initialise battery power as 0 incase load meets demand
        battery_power_kw = 0.0
        # initialise battery_soc_kwh and percent incase no battery detected on house these values are local only and the true values exist within the objects themselves
        battery_soc_kwh = 0.0
        battery_soc_percent = 0.0
        
        
        # check to see if there is a battery apart of this household
        if self.battery is not None:
            # check what mode the VPP controller is instructing and execute actions
            mode = signals["mode"]
            is_peak_period = signals["is_peak_period"]
            is_event_period = signals["is_event_period"]
            is_cheap_energy_period = signals["is_cheap_energy_period"]
            
            if mode == "self_consumption":
            
                if net_power_kw_bb < 0:
                # run charge function for power surplus
                    battery_power_kw = self.battery.charge(-net_power_kw_bb, timestep_hours)

                # else discharge battery to meet demand
                elif net_power_kw_bb > 0:
                    battery_power_kw = self.battery.discharge(net_power_kw_bb, timestep_hours)
            
               
            elif mode == "battery_hold":
                battery_power_kw = 0.0
                
            elif mode == "peak_shaving":
                # check to see if there is a surplus
                if net_power_kw_bb < 0:
                    # charge battery if there is a PV surplus
                    battery_power_kw = self.battery.charge(-net_power_kw_bb, timestep_hours)
                #Else check to see if it is within the peak period to shave demand
                elif net_power_kw_bb > 0 and is_peak_period:
                    # discharge battery if demand is greater than generation and within peak period time
                    battery_power_kw = self.battery.discharge(net_power_kw_bb, timestep_hours)
                else:
                    # battery is idle
                    battery_power_kw = 0.0
            elif mode == "export_support":
                battery_power_kw = self.battery.discharge(self.battery.max_discharge_kw, timestep_hours)
                # NOT USED CURRENTLY, ACTIVATE WHEN ECONOMIC MODEELING IS IMPLEMENTED
    
            elif mode == "local_load_support":
                target_kw = signals["house_dispatch_targets"].get(
                    self.name, 
                    0.0
                )

                battery_power_kw = self.battery.discharge(
                    target_kw,
                    timestep_hours
                )
                  
            elif mode == "arbitrage_charge":
                if is_cheap_energy_period:
                    battery_power_kw = self.battery.charge(self.battery.max_charge_kw, timestep_hours)
                
                
            else: 
                battery_power_kw = 0.0
            # update battery SOC values
            battery_soc_kwh = self.battery.get_soc_kwh()
            battery_soc_percent = self.battery.get_soc_percent()
            
        battery_discharge_kw = max(battery_power_kw, 0.0)
        battery_charge_kw = -min(battery_power_kw, 0.0)
        final_net_power_kw = load_kw - generation_kw - battery_power_kw   

        return {
            "name": self.name,
            "load_kw": load_kw,
            "generation_kw": generation_kw,
            "net_power_kw_before_battery": net_power_kw_bb,
            "battery_power_kw": battery_power_kw,
            "battery_soc_kwh": battery_soc_kwh,
            "battery_soc_percent": battery_soc_percent,
            "net_power_kw" : final_net_power_kw,
            "battery_charge_kw": battery_charge_kw,
            "battery_discharge_kw": battery_discharge_kw,
        }
#how do I improve load and generation simulation? need realistic load and genereation profiles for each house depending on their availiable resources. could also reseach files to obtain accurrate hourly load profiles (clean data in Excel)

# Create the Battery Class/Object, This may later be moved inside of the house class once tested for behaviour

class Battery:
    def __init__(self, capacity_kwh, initial_soc_kwh, max_charge_kw, max_discharge_kw, efficiency=0.95, min_soc_kwh=0.0):
        self.capacity_kwh = capacity_kwh
        self.soc_kwh = initial_soc_kwh
        self.max_charge_kw = max_charge_kw
        self.max_discharge_kw = max_discharge_kw
        self.efficiency = efficiency
        self.min_soc_kwh = min_soc_kwh
        
    def charge(self, requested_power_kw, timestep_hours):
        # check requested charge power is positive
        if requested_power_kw <= 0:
            return 0.0
        # limit requested power
        charge_power_kw = min(requested_power_kw, self.max_charge_kw)
        # convert to stored energy
        requested_energy_kwh = charge_power_kw *timestep_hours*self.efficiency
        # check availiable battery space
        available_space_kwh = self.capacity_kwh - self.soc_kwh
        # limit actual stored energy to within max capacity
        actual_energy_kwh = min(requested_energy_kwh, available_space_kwh)
        # update the SOC
        self.soc_kwh += actual_energy_kwh
        # calculate the charging power
        actual_charge_power_kw = actual_energy_kwh / (timestep_hours*self.efficiency)
        #return value of power stored by the battery to the simulation
        battery_power_kw = actual_charge_power_kw*-1
        return battery_power_kw
        
    def discharge(self, requested_power_kw, timestep_hours):
         # check requested discharge power is positive
        if requested_power_kw <= 0:
            return 0.0
        # limit requested discharge power
        discharge_power_kw = min(requested_power_kw, self.max_discharge_kw)
        # convert requested power to battery energy removed
        requested_energy_kwh = discharge_power_kw * timestep_hours / self.efficiency
        # check current SOC to determine if battery has available capacity
        available_energy_kwh = max(0.0, self.soc_kwh - self.min_soc_kwh)
        # limit discharged power to what is actually available
        actual_energy_kwh = min(available_energy_kwh, requested_energy_kwh) 
        # update the SOC
        self.soc_kwh -= actual_energy_kwh
        # calculate the discharge power
        actual_discharge_power_kw = (actual_energy_kwh*self.efficiency)/timestep_hours
        # return value of power discharged from the battery
        battery_power_kw = actual_discharge_power_kw
        return battery_power_kw
        
    def get_soc_percent(self):
        return (self.soc_kwh/self.capacity_kwh)*100
    def get_soc_kwh(self):
        return self.soc_kwh

# Define the VPP controller class
class VPPController:
    def __init__(self, 
                 default_mode="self_consumption", 
                 peak_hours=None, 
                 event_hours=None, 
                 cheap_energy_hours=None
                ):
        
        self.default_mode = default_mode
        self.peak_hours = peak_hours or []
        self.event_hours = event_hours or []
        self.cheap_energy_hours = cheap_energy_hours or []
        

    def get_signals(
        self, 
        t, 
        external_inputs=None, 
        house_states=None
    ):

        if external_inputs is None:
            external_inputs = {}

        if house_states is None:
            house_states = []


        network_support_request = external_inputs.get(
            "network_support_request",
            False
        )

        requested_support_kw = external_inputs.get(
            "requested_support_kw",
            0.0
        )

        house_dispatch_targets = {}

         # Calculate current VPP battery availability
        total_available_discharge_kw = sum(
            state["available_discharge_kw"]
            for state in house_states
        )
            

        manual_mode = external_inputs.get("manual_mode")

        if manual_mode is not None:
            active_mode = manual_mode
        
        elif network_support_request:
            active_mode = "local_load_support"

            available_houses = [
                state for state in house_states
                if state["has_battery"] 
                and state["available_discharge_kw"] > 0
            ]

            remaining_support_kw = requested_support_kw
    
            while available_houses and remaining_support_kw > 0:
    
                equal_share_kw = (remaining_support_kw/len(available_houses))
    
                still_available_houses = []
        
                for state in available_houses:
        
                    house_name = state["name"]
                    available_kw = state["available_discharge_kw"]
        
                   

                    already_dispatched_kw = (house_dispatch_targets.get(
                        house_name,
                        0.0
                        )
                    )

                    remaining_house_capacity_kw = (available_kw - already_dispatched_kw)

                    dispatch_kw = min(
                        equal_share_kw,
                        remaining_house_capacity_kw
                    )
        
                    house_dispatch_targets[house_name] = (already_dispatched_kw + dispatch_kw)
        
                    remaining_support_kw -= dispatch_kw
        
                    remaining_house_capacity_kw -= dispatch_kw
        
                    if remaining_house_capacity_kw > 0:
                        still_available_houses.append(state)
        
                available_houses = still_available_houses
        elif t in self.cheap_energy_hours:
            active_mode = "arbitrage_charge"
        elif t in self.peak_hours:
            active_mode = "peak_shaving"
        else:
            active_mode = self.default_mode

        return{
            "mode":active_mode,
            "is_peak_period":t in self.peak_hours,
            "is_event_period": network_support_request or t in self.event_hours,
            "is_cheap_energy_period":t in self.cheap_energy_hours,

            "requested_support_kw": requested_support_kw,
            "house_dispatch_targets": house_dispatch_targets,

            # Current VPP capability
            "total_available_discharge_kw": total_available_discharge_kw,
        }
        


#define a local load that the VPP can supply this is currently a basic place holder that will be updated later to match realistic demand values and have its own demand profile (currently static) 

class LocalLoad:
    def __init__(self, name, demand_kw):
        self.name = name
        self.demand_kw = demand_kw

    def step(self, t):
        return {
            "name": self.name,
            "load_kw": self.demand_kw[t]
        }


class WindTurbine:
    def __init__(self, name, rated_kw):
        self.name = name
        self.rated_kw = rated_kw

    def step(self, t):
        generation_kw = 0.0  # placeholder for now
        return {
            "name": self.name,
            "generation_kw": generation_kw
        }

#wind turbine could have three different generation settings based on the wind speed (no wind/lock out (too much wind), moderate wind, ideal wind)
#middle setting could have variance to show slight hour to hour changes perhaps?
#need to confirm how this works in realworld -> further research required



Simulation Loop

In [4]:
# PLACEHOLDER LOAD/GENERATION PROFILES UNTIL REAL VALUES ARE CLEANED AND SELECTED

# these load profiles will be replaced with real world data from the smart grid smart city dataset
local_loadevent = [6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 
                   6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 12.0, 12.0]   

house_1_load = [0.7, 0.6, 0.6, 0.5, 0.6, 0.8, 1.4, 1.8, 1.2, 1.0, 0.9, 0.8,
                0.9, 1.0, 1.1, 1.3, 1.8, 2.0, 2.4, 2.4, 1.8, 1.3, 1.0, 0.8]

house_2_load = [0.9, 0.8, 0.7, 0.7, 0.8, 1.0, 1.8, 2.2, 1.5, 1.2, 1.1, 1.0,
                1.0, 1.1, 1.3, 1.6, 2.2, 3.1, 2.4, 2.8, 2.1, 1.6, 1.2, 1.0]

house_3_load = [0.5, 0.5, 0.4, 0.4, 0.5, 0.6, 1.0, 1.3, 0.9, 0.8, 0.7, 0.7,
                0.8, 0.8, 0.9, 1.0, 1.4, 2.0, 2.2, 1.8, 1.3, 1.0, 0.8, 0.6]

house_4_load = [1.0, 0.9, 0.8, 0.8, 0.9, 1.2, 2.0, 2.5, 1.8, 1.5, 1.3, 1.2,
                1.3, 1.4, 1.6, 1.9, 2.6, 3.7, 3.0, 3.3, 2.5, 1.9, 1.4, 1.1]

house_5_load = [0.8, 0.7, 0.7, 0.6, 0.7, 0.9, 1.6, 2.0, 1.4, 1.1, 1.0, 0.9,
                1.0, 1.1, 1.2, 1.5, 2.0, 2.5, 2.8, 2.6, 2.0, 1.5, 1.1, 0.9]
# these generation profiles will be replaced with PV generation profiles generated using an external tool such as SAM to represent different size systems and weather conditions

pv_sunny_large = [0, 0, 0, 0, 0, 0.2, 0.8, 1.8, 3.0, 4.0, 4.8, 5.2,
             5.0, 4.5, 3.6, 2.4, 1.2, 0.3, 0, 0, 0, 0, 0, 0]

pv_sunny_med = [x*0.5 for x in pv_sunny_large]

pv_cloudy = [0, 0, 0, 0, 0, 0.1, 0.4, 0.9, 1.5, 2.0, 2.3, 2.5,
             2.2, 2.0, 1.6, 1.0, 0.5, 0.1, 0, 0, 0, 0, 0, 0]

pv_none = [0]*24

#these values will change later as the smart grid smart city dataset is in half hourly readings but overall goal is for each profile to simulate a full day to show arbitrage effect of peak hour demand

profile_duration_hours = 24 #this will probably remain unchanged 

# following code ensures calculations later in simulations remain correct if length of datasets change but each profile should alwuts represent 1 day
num_steps = len(house_1_load)
timestep_hours = profile_duration_hours / num_steps
simulation_hours = num_steps * timestep_hours

#need to add Wind Turbine and industrial load profiles here potentially? UPDATE WIND GEN AND LOCAL LOAD CLASS WHEN THIS IS DONE

# Verify all profiles are the same length
profiles = [
    house_1_load,
    house_2_load,
    house_3_load,
    house_4_load,
    house_5_load,
    pv_sunny_large,
    pv_sunny_med,
    pv_cloudy,
    pv_none]

profile_lengths = set([len(profile) for profile in profiles])


if len(profile_lengths) > 1 :
    raise ValueError("Profiles are not the same length")
else :
    print("All Load and Generation Profiles are the same length")

print("Number of Simulation steps=", num_steps)
print("Number of Hours Simulated=", simulation_hours) 

# Define battery size parameters for small, med and large classs batteries, this can later be matched to real battery models

SMALL_BATTERY = {
    "capacity_kwh": 5.0,
    "initial_soc_kwh": 0.5,
    "max_charge_kw": 2.0,
    "max_discharge_kw": 2.0,
    "efficiency": 0.95,
    "min_soc_kwh": 0.5
}

MEDIUM_BATTERY = {
    "capacity_kwh": 10.0,
    "initial_soc_kwh": 1.0,
    "max_charge_kw": 4.0,
    "max_discharge_kw": 4.0,
    "efficiency": 0.95,
    "min_soc_kwh": 1.0
}

LARGE_BATTERY = {
    "capacity_kwh": 15.0,
    "initial_soc_kwh": 1.5,
    "max_charge_kw": 5.0,
    "max_discharge_kw": 5.0,
    "efficiency": 0.95,
    "min_soc_kwh": 1.5
}



All Load and Generation Profiles are the same length
Number of Simulation steps= 24
Number of Hours Simulated= 24.0


In [28]:

# initialise the VPP controller to test all modes
# initialise the VPP controller
# controller = VPPController(
#     default_mode="self_consumption",
#     peak_hours=[],
#     event_hours=[],
#     cheap_energy_hours=[]
# )


# initialise the VPP controller
# peak shaving default mode allows for forward planning with peak shaving mode
controller = VPPController(
    default_mode="self_consumption",
    peak_hours=[],
    event_hours=[],
    cheap_energy_hours=[]
)
scenario_name="external_network_support_test"


# Initialise Batteries
house1_battery = Battery(**LARGE_BATTERY)
house2_battery = Battery(**MEDIUM_BATTERY)
house4_battery = Battery(**MEDIUM_BATTERY)
house5_battery = Battery(**LARGE_BATTERY)
# Initialise Houses
houses = [
    House("House 1", house_1_load, pv_sunny_large, house1_battery),
    House("House 2", house_2_load, pv_sunny_med, house2_battery),
    House("House 3", house_3_load, pv_sunny_large),
    House("House 4", house_4_load, pv_sunny_large, house4_battery),
    House("House 5", house_5_load, pv_none, house5_battery),
]


# Initialise hospital and wind turbine

hospital = LocalLoad("Hospital", local_loadevent)
wind = WindTurbine("DFIG Wind Placeholder", 15.0)


# Create simulation
simulation = VPPSimulation(
    houses=houses,
    hospital=hospital,
    wind=wind,
    controller=controller,
    timestep_hours=timestep_hours,
    num_steps=num_steps
)



test_days = 5
steps_to_run = int((24 / timestep_hours) * test_days)

event_day = 3
event_hour = 14
event_duration_hours = 2

event_start_step = int(
    (((event_day - 1) * 24) + event_hour)
    / timestep_hours
)

event_duration_steps = int(
    event_duration_hours / timestep_hours
)

event_end_step = (
    event_start_step
    + event_duration_steps
)


# for _ in range(steps_to_run):

#     external_inputs = {}

#     if (
#         event_start_step
#         <= simulation.current_step
#         < event_end_step
#     ):
#         external_inputs = {
#             "network_support_request": True,
#             "requested_support_kw": 8.0
#         }

#     simulation.step(external_inputs)

for _ in range(steps_to_run):

    external_inputs = {}

    if (
        event_start_step
        <= simulation.current_step
        < event_end_step
    ):
        external_inputs = {
            "network_support_request": True,
            "requested_support_kw": 28.0
        }

    result = simulation.step(external_inputs)

    if result["VPP_mode"] == "local_load_support":
        print(
            "Step:",
            result["Simulation_step"],
            "Mode:",
            result["VPP_mode"],
            "Requested:",
            result["Requested_support_kw"],
            "Available:",
            result["Available_VPP_discharge_kw"],
            "Actual discharge:",
            result["Battery_discharge_kw"]
        )
        print("Individual house discharge:")

        for house_name, house_log in simulation.house_results_log.items():

            latest_house_result = house_log[-1]

            print(
                house_name,
                ":",
                latest_house_result["battery_discharge_kw"],
                "kW"
            )


# old test code

# test_days = 5
# steps_to_run = int((24 / timestep_hours) * test_days)

# for _ in range(steps_to_run):
#     simulation.step()
    


# profile_index = simulation.current_step % simulation.num_steps

# for house in houses:
#     state = house.get_state(
#         profile_index,
#         timestep_hours
#     )

#     print(state)


    
# Test external input overide 
# for _ in range(steps_to_run):

#     if simulation.current_step == 22:
#         simulation.step({
#             "manual_mode":"battery_hold"
#                         })
#     else:
#         simulation.step()

# test_result = simulation.simulation_results_summary[22]

# print("Step:", test_result["Simulation_step"])
# print("Mode:", test_result["VPP_mode"])
# print("Battery Power:", test_result["Battery_power_kw"])
# next_result = simulation.simulation_results_summary[23]

# print("Step:", next_result["Simulation_step"])
# print("Mode:", next_result["VPP_mode"])
# print("Battery Power:", next_result["Battery_power_kw"])





timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

filename = f"SimulationResults_{scenario_name}_{timestamp}.xlsx"
filepath = f"results/{filename}"


summary_df = pd.DataFrame(
    simulation.simulation_results_summary
).round(2)


with pd.ExcelWriter(filepath, engine="openpyxl") as writer:

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for house_name, house_log in simulation.house_results_log.items():

        house_df = pd.DataFrame(house_log).round(2)

        sheet_name = house_name.replace(" ", "_")

        house_result_export = house_df.drop(
            columns=["name"]
        )

        house_result_export.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )


print(
    simulation.current_step,
    "simulation steps completed"
)

print(
    f"Results exported to results/{filename}"
)

Step: 62 Mode: local_load_support Requested: 28.0 Available: 13.0 Actual discharge: 13.0
Individual house discharge:
House 1 : 5.0 kW
House 2 : 3.9999999999999996 kW
House 3 : 0.0 kW
House 4 : 3.9999999999999996 kW
House 5 : 0.0 kW
Step: 63 Mode: local_load_support Requested: 28.0 Available: 10.730875000000001 Actual discharge: 10.730875000000001
Individual house discharge:
House 1 : 5.0 kW
House 2 : 1.7308750000000008 kW
House 3 : 0.0 kW
House 4 : 3.9999999999999996 kW
House 5 : 0.0 kW
120 simulation steps completed
Results exported to results/SimulationResults_external_network_support_test_2026-09-01_21-40.xlsx


In [ ]:
# # battery class testing
# battery_test = Battery(
#     capacity_kwh=10,
#     initial_soc_kwh=5.0,
#     max_charge_kw=2.0,
#     max_discharge_kw=2.0,
#     efficiency=0.95,
#     min_soc_kwh=1.0
# )



# timestep_hours = 1.0

# print(round(battery_test.get_soc_percent(),2),"%")
# charge_test = battery_test.charge(2.0, timestep_hours)
# print(round(charge_test,2))
# print(round(battery_test.get_soc_percent(),2),"%")


# discharge_test=battery_test.discharge(4.0, timestep_hours)
# print(round(discharge_test,2))
# print(round(battery_test.get_soc_percent(),2),"%")


